In [1]:
import os
import math
import random
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Reduces allocator fragmentation -- set before any CUDA allocation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# -- FLAGS --
RESUME_TRAINING = False  # True  -> load latest/best checkpoint and continue
                         # False -> start from scratch
VRAM_FRACTION   = 0.85   # fraction of VRAM PyTorch may use (0.85 = 10.2 GB on 12 GB)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.cuda.set_per_process_memory_fraction(VRAM_FRACTION, device=0)
    print(f'Using device: {device}  |  VRAM cap: {VRAM_FRACTION*100:.0f}%')
else:
    print('Using device: cpu')

DATA_ROOT  = os.path.join(os.getcwd(), 'data')
MODEL_ROOT = os.path.join(os.getcwd(), 'models')
os.makedirs(MODEL_ROOT, exist_ok=True)

Using device: cuda  |  VRAM cap: 85%


In [2]:
from helper_functions.utils import MARSDataset

MARS_ROOT = os.path.join(DATA_ROOT, 'MARS')

dataset_train = MARSDataset(os.path.join(MARS_ROOT, 'bounding_box_train'), min_seq_len=2)
dataset_test  = MARSDataset(os.path.join(MARS_ROOT, 'bounding_box_test'),  min_seq_len=1)

all_seqs_train = [s for cams in dataset_train.sequences_by_person.values()
                    for seqs in cams.values() for s in seqs]
all_seqs_test  = [s for cams in dataset_test.sequences_by_person.values()
                    for seqs in cams.values() for s in seqs]

print(f'Train identities: {len(dataset_train.sequences_by_person)}')
print(f'Train sequences:  {len(all_seqs_train)}')
print(f'Test  identities: {len(dataset_test.sequences_by_person)}')
print(f'Test  sequences:  {len(all_seqs_test)}')

Train identities: 625
Train sequences:  8298
Test  identities: 634
Test  sequences:  8062


In [3]:
import numpy as np

lens = [len(s) for s in all_seqs_train]
print(f'Train seq lengths -- mean: {np.mean(lens):.1f}  median: {np.median(lens):.0f}  '
      f'p95: {np.percentile(lens,95):.0f}  p99: {np.percentile(lens,99):.0f}  max: {max(lens)}')

Train seq lengths -- mean: 61.5  median: 36  p95: 186  p99: 277  max: 900


In [ ]:
import torchvision.transforms as T
from torch.utils.data import Sampler

# MARS tracklets are 25-50 frames; cap at 16 to stay within VRAM budget.
MAX_SEQ_LEN = 32
P, K = 8, 3  # 48 sequences/batch

train_transform = T.Compose([
    T.Resize((256, 128)),
    T.RandomHorizontalFlip(),
    T.Pad(10),
    T.RandomCrop((256, 128)),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3)),
])

test_transform = T.Compose([
    T.Resize((256, 128)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class SequenceLabelDataset(Dataset):
    """Returns (frames, length, person_id)."""
    def __init__(self, sequences_by_person, transform, max_seq_len):
        self.transform          = transform
        self.max_seq_len        = max_seq_len
        self.samples            = []
        self.pid_to_indices     = {}
        self.pid_cam_to_indices = {}

        for pid, cam_data in sequences_by_person.items():
            for cam, seqs in cam_data.items():
                for seq in seqs:
                    i = len(self.samples)
                    self.samples.append((seq, pid))
                    self.pid_to_indices.setdefault(pid, []).append(i)
                    self.pid_cam_to_indices.setdefault(pid, {}).setdefault(cam, []).append(i)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq, pid = self.samples[idx]
        paths  = [p.image_path for p in seq[:self.max_seq_len]]
        frames = [self._load(p) for p in paths]
        return torch.stack(frames), len(frames), pid

    def _load(self, path):
        return self.transform(Image.open(path).convert('RGB'))


class PKSampler(Sampler):
    """Camera-aware PK sampler -- prefers cross-camera sequences per person."""
    def __init__(self, pid_to_indices, pid_cam_to_indices, P=12, K=4):
        self.pid_to_indices     = pid_to_indices
        self.pid_cam_to_indices = pid_cam_to_indices
        self.P = P
        self.K = K
        self.pids = [pid for pid, idxs in pid_to_indices.items() if len(idxs) >= 2]

    def _sample_cross_camera(self, pid):
        cam_data = self.pid_cam_to_indices[pid]
        cameras  = list(cam_data.keys())
        if len(cameras) == 1:
            return random.choices(self.pid_to_indices[pid], k=self.K)
        random.shuffle(cameras)
        cam_cycle = cameras * (self.K // len(cameras) + 1)
        return [random.choice(cam_data[cam]) for cam in cam_cycle[:self.K]]

    def __iter__(self):
        pids = self.pids.copy()
        random.shuffle(pids)
        for i in range(0, len(pids) - self.P + 1, self.P):
            batch = []
            for pid in pids[i : i + self.P]:
                batch.extend(self._sample_cross_camera(pid))
            yield batch

    def __len__(self):
        return len(self.pids) // self.P


def sequence_label_collate(batch):
    imgs_list, lengths, pids = zip(*batch)
    max_t = max(lengths)
    C, H, W = imgs_list[0].shape[1:]
    padded = torch.zeros(len(imgs_list), max_t, C, H, W)
    for i, (imgs, t) in enumerate(zip(imgs_list, lengths)):
        padded[i, :t] = imgs
    return padded, torch.tensor(lengths, dtype=torch.long), torch.tensor(pids, dtype=torch.long)

In [5]:
train_dataset = SequenceLabelDataset(
    dataset_train.sequences_by_person,
    transform=train_transform,
    max_seq_len=MAX_SEQ_LEN
)

sampler = PKSampler(train_dataset.pid_to_indices,
                    train_dataset.pid_cam_to_indices, P=P, K=K)

print(f'Train sequences:       {len(train_dataset)}')
print(f'Persons with >=2 seqs: {len(sampler.pids)}')
print(f'Steps per epoch:       {len(sampler)}')

train_loader = DataLoader(
    train_dataset,
    batch_sampler=sampler,
    collate_fn=sequence_label_collate,
    pin_memory=True,
    num_workers=0
)

Train sequences:       8298
Persons with >=2 seqs: 624
Steps per epoch:       78


In [6]:
from helper_functions.model import ImprovedSeqToSeqReIDModel, CombinedReIDLoss

all_train_pids = sorted(dataset_train.sequences_by_person.keys())
pid_to_cls     = {pid: i for i, pid in enumerate(all_train_pids)}
NUM_CLASSES    = len(all_train_pids)
print(f'Training identities: {NUM_CLASSES}')

model = ImprovedSeqToSeqReIDModel(
    embedding_dim=512,
    rnn_hidden=512,
    num_classes=NUM_CLASSES
).to(device)

# Layer-wise LR decay across all 12 ViT blocks
vit_base_lr = 5e-6
decay       = 0.75

param_groups = []
param_groups.append({
    'params': list(model.encoder.vit.patch_embed.parameters()) +
              [model.encoder.vit.pos_embed, model.encoder.vit.cls_token],
    'lr': vit_base_lr * (decay ** 12)
})
for i, block in enumerate(model.encoder.vit.blocks):
    param_groups.append({'params': list(block.parameters()), 'lr': vit_base_lr * (decay ** (11 - i))})
param_groups.append({'params': list(model.encoder.vit.norm.parameters()), 'lr': vit_base_lr})

head_params = (
    list(model.encoder.global_proj.parameters()) +
    list(model.encoder.local_proj.parameters()) +
    list(model.encoder.temporal_tf.parameters()) +
    list(model.encoder.rnn.parameters()) +
    list(model.encoder.attention.parameters()) +
    list(model.encoder.embed_head.parameters()) +
    (list(model.classifier.parameters()) if model.classifier is not None else [])
)
param_groups.append({'params': head_params, 'lr': 5e-5})

optimizer = torch.optim.Adam(param_groups)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-8, verbose=True
)
scaler    = torch.amp.GradScaler('cuda')
criterion = CombinedReIDLoss(margin=0.3, circle_m=0.25, circle_gamma=80, lambda_circle=0.5).to(device)

BEST_CKPT   = os.path.join(MODEL_ROOT, 'best_model_mars.pth')
LATEST_CKPT = os.path.join(MODEL_ROOT, 'latest_model_mars.pth')

if RESUME_TRAINING:
    for ckpt in [LATEST_CKPT, BEST_CKPT]:
        if os.path.exists(ckpt):
            model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
            print(f'Resumed from {os.path.basename(ckpt)}')
            break
    else:
        print('WARNING: RESUME_TRAINING=True but no checkpoint found. Starting fresh.')
else:
    print('Starting from scratch.')

c:\Users\jokub\miniforge3\envs\tiriamasis\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training identities: 625


c:\Users\jokub\miniforge3\envs\tiriamasis\lib\site-packages\torch\nn\modules\module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\cb\pytorch_1000000000000\work\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


Starting from scratch.


c:\Users\jokub\miniforge3\envs\tiriamasis\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [7]:
def train_epoch(model, loader, optimizer, scaler, device, pid_to_cls, criterion):
    model.train()
    total_loss = 0.0
    n = 0

    for imgs, lengths, pids in tqdm(loader, desc='Train'):
        imgs = imgs.to(device)
        pids = pids.to(device)
        cls_labels = torch.tensor([pid_to_cls[p.item()] for p in pids],
                                   dtype=torch.long, device=device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            embeddings, logits = model(imgs, lengths)
            loss, _ = criterion(embeddings, cls_labels, logits)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        n += 1

    return total_loss / n if n > 0 else 0.0

In [8]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
import gc

def encode_sequences(sequences, model, transform, device,
                      max_seq_len=None, batch_size=32, flip_tta=False):
    model.eval()

    def seq_len(seq):
        s = seq if max_seq_len is None else seq[:max_seq_len]
        return len(s)

    sequences = sorted(sequences, key=seq_len)
    all_embs, all_pids = [], []

    def load_seq(seq, flipped=False):
        frames_to_use = seq if max_seq_len is None else seq[:max_seq_len]
        frames = []
        for p in frames_to_use:
            img = Image.open(p.image_path).convert('RGB')
            if flipped:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
            frames.append(transform(img))
        return torch.stack(frames), len(frames), seq[0].person_id

    def run_batch(batch_seqs, flipped=False):
        results = [load_seq(s, flipped) for s in batch_seqs]
        imgs_list, lengths, pids = zip(*results)
        max_t = max(lengths)
        C, H, W = imgs_list[0].shape[1:]
        padded = torch.zeros(len(imgs_list), max_t, C, H, W)
        for i, (imgs, t) in enumerate(zip(imgs_list, lengths)):
            padded[i, :t] = imgs
        lengths_t = torch.tensor(lengths, dtype=torch.long)
        with torch.amp.autocast('cuda'):
            embs = model(padded.to(device), lengths_t)
        return embs.cpu().float(), pids

    with torch.no_grad():
        for start in tqdm(range(0, len(sequences), batch_size), desc='Encoding'):
            batch_seqs = sequences[start : start + batch_size]
            embs, pids = run_batch(batch_seqs)
            if flip_tta:
                embs_flip, _ = run_batch(batch_seqs, flipped=True)
                import torch.nn.functional as F
                embs = F.normalize((embs + embs_flip) / 2, dim=1)
            all_embs.append(embs.numpy())
            all_pids.extend(pids)

    return np.concatenate(all_embs, axis=0), np.array(all_pids)


def evaluate(query_embs, query_pids, gallery_embs, gallery_pids):
    sim        = cos_sim(query_embs, gallery_embs)
    sorted_idx = np.argsort(-sim, axis=1)
    rank1 = np.mean(gallery_pids[sorted_idx[:, 0]] == query_pids)
    aps = []
    for i in range(len(query_pids)):
        relevant = gallery_pids[sorted_idx[i]] == query_pids[i]
        if relevant.sum() == 0:
            continue
        cum  = np.cumsum(relevant)
        prec = cum / (np.arange(1, len(relevant) + 1))
        aps.append(prec[relevant].mean())
    return rank1, float(np.mean(aps)) if aps else 0.0

In [9]:
# MARS has no official query folder -- build cross-camera split from bounding_box_test.
# Query:   first tracklet per person per camera
# Gallery: all tracklets from DIFFERENT cameras for the same person

query_seqs, gallery_seqs = [], []

for pid, cam_data in dataset_test.sequences_by_person.items():
    cameras = list(cam_data.keys())
    if len(cameras) < 2:
        # Only one camera -- add all to gallery, skip as query
        for seqs in cam_data.values():
            gallery_seqs.extend(seqs)
        continue
    query_cam = cameras[0]
    for cam, seqs in cam_data.items():
        if cam == query_cam:
            query_seqs.append(seqs[0])   # one query tracklet per person
        else:
            gallery_seqs.extend(seqs)    # all other-camera tracklets as gallery

print(f'Query sequences:   {len(query_seqs)}')
print(f'Gallery sequences: {len(gallery_seqs)}')

Query sequences:   626
Gallery sequences: 5853


In [10]:
NUM_EPOCHS = 100
PATIENCE   = 10

best_rank1       = 0.0
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, scaler,
                             device, pid_to_cls, criterion)

    torch.save(model.state_dict(), LATEST_CKPT)

    gc.collect()
    torch.cuda.empty_cache()
    q_embs, q_pids = encode_sequences(query_seqs,   model, test_transform, device, flip_tta=True)
    g_embs, g_pids = encode_sequences(gallery_seqs, model, test_transform, device,
                                       max_seq_len=MAX_SEQ_LEN, flip_tta=True)
    model.train()
    gc.collect()
    torch.cuda.empty_cache()
    rank1, mAP = evaluate(q_embs, q_pids, g_embs, g_pids)

    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}]  loss={train_loss:.4f}  Rank-1={rank1:.4f}  mAP={mAP:.4f}')

    scheduler.step(rank1)

    if rank1 > best_rank1:
        best_rank1       = rank1
        patience_counter = 0
        torch.save(model.state_dict(), BEST_CKPT)
        print('  --> Saved best model')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print('Early stopping.')
            break

print(f'Best Rank-1: {best_rank1:.4f}')

Train:   0%|          | 0/78 [00:16<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 734.00 MiB. GPU 0 has a total capacity of 12.00 GiB of which 0 bytes is free. 10.20 GiB allowed; Of the allocated memory 9.42 GiB is allocated by PyTorch, and 348.02 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

model.load_state_dict(torch.load(BEST_CKPT, map_location=device, weights_only=True))

q_embs, q_pids = encode_sequences(query_seqs,   model, test_transform, device, flip_tta=True)
g_embs, g_pids = encode_sequences(gallery_seqs, model, test_transform, device,
                                   max_seq_len=MAX_SEQ_LEN, flip_tta=True)
rank1, mAP = evaluate(q_embs, q_pids, g_embs, g_pids)


def re_ranking(q_embs, g_embs, k1=20, k2=6, lambda_value=0.3):
    feat = np.concatenate([q_embs, g_embs], axis=0)
    n_q, n = len(q_embs), len(q_embs) + len(g_embs)
    dist = 2 - 2 * (feat @ feat.T).clip(-1, 1)
    initial_rank = np.argsort(dist, axis=1)
    V = np.zeros_like(dist)
    for i in range(n):
        forward = set(initial_rank[i, 1:k1+1])
        R_i = [j for j in forward if i in set(initial_rank[j, 1:k1+1])]
        R_i_exp = list(R_i)
        for q in R_i:
            R_q = set(initial_rank[q, 1:int(k1/2)+1])
            if len(R_q) and len(R_q & set(R_i)) / len(R_q) >= 2/3:
                R_i_exp += list(R_q)
        R_i_exp = list(set(R_i_exp))
        w = np.exp(-dist[i, R_i_exp])
        w /= w.sum() + 1e-12
        V[i, R_i_exp] = w
    if k2 != 1:
        V_qe = np.zeros_like(V)
        for i in range(n):
            V_qe[i] = V[initial_rank[i, :k2]].mean(axis=0)
        V = V_qe
    jaccard = np.zeros((n_q, n - n_q))
    for i in range(n_q):
        for j in range(n_q, n):
            inter = np.minimum(V[i], V[j]).sum()
            union = np.maximum(V[i], V[j]).sum()
            jaccard[i, j - n_q] = 1 - inter / (union + 1e-12)
    return jaccard * (1 - lambda_value) + dist[:n_q, n_q:] * lambda_value


def evaluate_rerank(q_embs, q_pids, g_embs, g_pids, k1=20, k2=6, lambda_value=0.3):
    dist = re_ranking(q_embs, g_embs, k1=k1, k2=k2, lambda_value=lambda_value)
    sorted_idx = np.argsort(dist, axis=1)
    rank1_rr = np.mean(g_pids[sorted_idx[:, 0]] == q_pids)
    aps = []
    for i in range(len(q_pids)):
        relevant = g_pids[sorted_idx[i]] == q_pids[i]
        if relevant.sum() == 0:
            continue
        cum  = np.cumsum(relevant)
        prec = cum / (np.arange(1, len(relevant) + 1))
        aps.append(prec[relevant].mean())
    return rank1_rr, float(np.mean(aps)) if aps else 0.0


rank1_rr, mAP_rr = evaluate_rerank(q_embs, q_pids, g_embs, g_pids)
print(f'Without re-ranking: Rank-1={rank1:.4f}  mAP={mAP:.4f}')
print(f'With    re-ranking: Rank-1={rank1_rr:.4f}  mAP={mAP_rr:.4f}')

sim        = cos_sim(q_embs, g_embs)
sorted_idx = np.argsort(-sim, axis=1)
MAX_RANK   = 20
cmc = np.zeros(MAX_RANK)
for i in range(len(q_pids)):
    matches = g_pids[sorted_idx[i]] == q_pids[i]
    for r in range(MAX_RANK):
        if matches[:r + 1].any():
            cmc[r:] += 1
            break
cmc /= len(q_pids)

plt.figure(figsize=(8, 5))
plt.plot(range(1, MAX_RANK + 1), cmc, marker='o', linewidth=2, markersize=4)
plt.xlabel('Rank')
plt.ylabel('Identification Rate')
plt.title('CMC Curve -- MARS')
plt.xticks(range(1, MAX_RANK + 1))
plt.ylim(0, 1)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('mars_cmc.png', dpi=150)
plt.show()

print(f'Rank-1={cmc[0]:.4f}  Rank-5={cmc[4]:.4f}  Rank-10={cmc[9]:.4f}  Rank-20={cmc[19]:.4f}  mAP={mAP:.4f}')

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import matplotlib.cm as cm

N_PERSONS     = 20
unique_pids   = np.unique(q_pids)
selected_pids = unique_pids[:N_PERSONS]

mask     = np.isin(q_pids, selected_pids)
embs_sub = q_embs[mask]
pids_sub = q_pids[mask]

tsne   = TSNE(n_components=2, perplexity=min(30, len(embs_sub) - 1),
              n_iter_without_progress=1000, random_state=42)
coords = tsne.fit_transform(embs_sub)

colors    = cm.tab20(np.linspace(0, 1, N_PERSONS))
pid_color = {pid: colors[i] for i, pid in enumerate(selected_pids)}

plt.figure(figsize=(10, 8))
for pid in selected_pids:
    idx = pids_sub == pid
    plt.scatter(coords[idx, 0], coords[idx, 1],
                color=pid_color[pid], label=f'ID {pid}', s=45, alpha=0.85)

plt.legend(loc='best', fontsize=6, ncol=2, framealpha=0.7)
plt.title('t-SNE -- Query Embeddings (MARS)')
plt.axis('off')
plt.tight_layout()
plt.savefig('mars_tsne.png', dpi=150)
plt.show()